In [ ]:
from pathlib import Path

import pandas as pd

from config.config import REGISTRIES_DIR, SUBREGISTRIES_DIR, VersionObject

DOCUMENT_REGISTRY: Path =  REGISTRIES_DIR / "document_registry.csv"
BASE_REGISTRY: Path = REGISTRIES_DIR / "base_output_registry.csv"
RAW_REGISTRY: Path = SUBREGISTRIES_DIR / "raw_file_output_registry.csv"
EXTRACTION_REGISTRY: Path = SUBREGISTRIES_DIR / "extraction_output_registry.csv"
CLASSIFICATION_REGISTRY: Path = SUBREGISTRIES_DIR / "DAS_classification_output_registry.csv"
EMBEDDING_REGISTRY: Path = SUBREGISTRIES_DIR / "embedding_output_registry.csv"

version_object = VersionObject(pipeline_version="v1.0.0", software_version="v1.0.3")
df_document_registry = pd.read_csv(DOCUMENT_REGISTRY)
df_base_registry = pd.read_csv(BASE_REGISTRY)
df_raw_registry = pd.read_csv(RAW_REGISTRY)
df_extraction_registry = pd.read_csv(EXTRACTION_REGISTRY)
df_classification_registry = pd.read_csv(CLASSIFICATION_REGISTRY)
df_embedding_registry = pd.read_csv(EMBEDDING_REGISTRY)

In [ ]:
"""embedding computation for SentenceTransformer library"""
import logging
from datetime import datetime

import numpy as np
from sentence_transformers import SentenceTransformer

from application.utils import compute_hashes, single_line_text
from config.config import RECORDS_DIR

compute_embeddings_logger = logging.getLogger()

def compute_embeddings_single_file(
    cleaned_section_path: str,
    model_name: str,
    doc_doi: str,
    folder_doi: str,
    df_base: pd.DataFrame,
    df_embedding: pd.DataFrame,
    version_object: VersionObject,
    dependency_set: set,
    force_processing: bool = False
):

    compute_embeddings_logger.info("Starting to compute embeddings...")

    dependency_sha = compute_hashes(cleaned_section_path, salt=f"cleaned:{doc_doi}_{version_object.software_version}")

    if force_processing or dependency_sha not in dependency_set: 

        section_prefix = "DAS" if "DAS" in str(cleaned_section_path) else "CAS"

        embedding_path = Path(
            RECORDS_DIR
            / version_object.software_version
            / "pipeline_version"
            / version_object.pipeline_version
            / folder_doi
            / f"{section_prefix}_embedding.npy"
        )

        try:
            dependency_sha = compute_hashes(cleaned_section_path, salt=f"cleaned:{doc_doi}_{version_object.software_version}")

            text = cleaned_section_path.read_text()

            text_single_line = single_line_text(text)

            model = SentenceTransformer(model_name)

            embedding = model.encode(text_single_line)

            with open(embedding_path, "wb") as f:
                np.save(f, embedding)

                output_sha = compute_hashes(embedding_path, salt=doc_doi)

                base_row = {
                    "output_sha": output_sha,
                    "doc_doi": doc_doi,
                    "output_type": "embedding",
                    "pipeline_version": version_object.pipeline_version,
                    "software_version": version_object.software_version,
                    "creation_date": datetime.now(),
                    "dependencies": dependency_sha,
                }

                embedding_row = {"output_sha": output_sha, "file_path": embedding_path, "model": model_name}

                df_base = pd.concat([df_base, pd.DataFrame([base_row])], ignore_index=True)

                df_embedding = pd.concat([df_embedding, pd.DataFrame([embedding_row])], ignore_index=True)

                return df_base, df_embedding, dependency_sha

        except (FileNotFoundError, OSError, PermissionError):
            compute_embeddings_logger.exception("An error occurred while computing embeddings")

        else:

            return df_base, df_embedding, dependency_sha

In [ ]:
from tqdm import tqdm

from application.utils import save_registry


def compute_embeddings(df_base: pd.DataFrame, df_extraction: pd.DataFrame, df_embedding: pd.DataFrame, version_object: VersionObject, section_type: list | None = None, force_processing: bool = False, model_name: str = "sentence-transformers/all-MiniLM-L6-v2") -> tuple[pd.DataFrame, pd.DataFrame]:

    merged_base_extraction = df_extraction.merge(df_base[["output_sha", "doc_doi", "software_version"]], on="output_sha", how="left")

    dependency_set = set(
        df_base.loc[(df_base["output_type"]=="embedding")
                    & (df_base["software_version"]==version_object.software_version),
                    "dependencies"
        ].dropna()
    )

    rows = merged_base_extraction.loc[(merged_base_extraction["section_type"].isin(section_type)) & (merged_base_extraction["stage"] == "cleaned") & (merged_base_extraction["software_version"] == version_object.software_version)]

    for _, row in tqdm(rows.iterrows(), total=len(rows), desc=f"Embedding {section_type}", unit="section(s)"):
    
        path = Path(row["file_path"])

        doc_doi = row["doc_doi"]

        folder_doi = doc_doi.replace("/", "_")

        df_base, df_embedding, dependency_sha = compute_embeddings_single_file(
            path,
            model_name,
            doc_doi,
            folder_doi,
            df_base,
            df_embedding,
            version_object,
            dependency_set,
            force_processing=force_processing)
        
        dependency_set.add(dependency_sha)

    save_registry(df_base, REGISTRIES_DIR / "base_output_registry.csv")

    save_registry(df_embedding, SUBREGISTRIES_DIR / "embedding_output_registry.csv")

    return df_base, df_embedding



In [ ]:
print(df_extraction_registry["section_type"].unique())

merged_base_extraction = df_extraction_registry.merge(df_base_registry[["output_sha", "doc_doi", "software_version"]], on="output_sha", how="left")

print(merged_base_extraction["stage"].unique())

print(merged_base_extraction.columns)

print(merged_base_extraction["doc_doi"].isna().sum())

print("Total rows after merge:", len(merged_base_extraction))

print("Section types:", merged_base_extraction["section_type"].unique())
print("Stages:", merged_base_extraction["stage"].unique())
print("Software versions:", merged_base_extraction["software_version"].unique())

section_type = ["CAS"]

rows = merged_base_extraction.loc[(merged_base_extraction["section_type"]=="CAS") & (merged_base_extraction["stage"] == "cleaned") & (merged_base_extraction["software_version"] == version_object.software_version)]

print("Filtered rows:", len(rows))

In [ ]:
merged = df_extraction_registry.merge(
    df_base_registry[["output_sha", "doc_doi", "software_version"]], 
    on="output_sha", 
    how="left"
)

nan_rows = merged[merged["doc_doi"].isna()]
print(f"Total rows with NaN doc_doi: {len(nan_rows)}")
print(nan_rows[["output_sha", "section_type", "stage"]].head(10))

# check if those output_shas exist in base registry at all
missing_in_base = nan_rows[~nan_rows["output_sha"].isin(df_base_registry["output_sha"])]
print(f"\nSHAs completely missing from base registry: {len(missing_in_base)}")

# check if they exist but with NaN doc_doi
present_but_nan = nan_rows[nan_rows["output_sha"].isin(df_base_registry["output_sha"])]
print(f"SHAs present in base registry but doc_doi is NaN: {len(present_but_nan)}")

In [ ]:
merged_base_extraction = df_extraction_registry.merge(
    df_base_registry[["output_sha", "doc_doi", "software_version"]], 
    on="output_sha", 
    how="left"
)

rows = merged_base_extraction.loc[
    (merged_base_extraction["section_type"].isin(["CAS"])) & 
    (merged_base_extraction["stage"] == "cleaned") & 
    (merged_base_extraction["software_version"] == version_object.software_version)
]

print(rows["doc_doi"].isna().sum())
print(rows[rows["doc_doi"].isna()][["output_sha", "section_type", "stage", "file_path"]])

rows.tail(10)

In [ ]:
df_base_registry, df_embedding_registry = compute_embeddings(df_base_registry, df_extraction_registry, df_embedding_registry, version_object, section_type=["CAS"])

In [ ]:
print(type(RECORDS_DIR))
print(RECORDS_DIR)